# Skills Gap Analyzer — Optimized v2

Key improvements over v1:
- Fixed `is_negated()` — now uses window-based search, not fragile positional logic
- Added **degradation / comparative negative** phrases (too slow, took too long, below expectations, etc.)
- Clause splitting now tags clauses with their **contrast polarity** (after 'but', 'however' = likely negative)
- Gap detection uses clause polarity context, not just keyword presence
- Removed false positive inflation from over-splitting

In [ ]:
!pip install sentence-transformers wtpsplit -q

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.9/152.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 16.2 MB/s eta 0:00:00


In [ ]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1")
print("Model loaded")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/677 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Model loaded


In [ ]:
SKILL_ALIASES = {

    # ── LEADERSHIP ──────────────────────────────────────────────
    "leadership": [
        "leadership", "leads the team", "mentors others",
        "guides colleagues", "takes charge", "inspires others",
        "sets direction", "drives the team", "takes ownership"
    ],
    "mentoring": [
        "mentoring", "coaches others", "develops team members",
        "supports junior staff", "guides new hires", "teaches colleagues"
    ],
    "delegation": [
        "delegation", "assigns tasks effectively", "distributes work",
        "trusts team with responsibilities", "empowers others"
    ],
    "decision making": [
        "decision making", "makes sound decisions", "decides confidently",
        "exercises good judgment", "thinks before acting", "evaluates options"
    ],
    "accountability": [
        "accountability", "takes responsibility", "owns mistakes",
        "follows through", "delivers on commitments", "reliable"
    ],
    "initiative": [
        "initiative", "proactive", "self-starter", "acts without being told",
        "takes the lead", "goes beyond expectations", "drives change"
    ],

    # ── COMMUNICATION ────────────────────────────────────────────
    "communication": [
        "communication", "articulates clearly", "expresses ideas well",
        "conveys information", "speaks confidently", "writes clearly"
    ],
    "availability": [
        "availability", "always available", "reachable when needed",
        "present when needed", "responsive to team"
    ],
    "active listening": [
        "active listening", "listens carefully", "pays attention",
        "understands others", "hears feedback well", "attentive in meetings"
    ],
    "presentation": [
        "presentation", "presents well", "delivers engaging talks",
        "public speaking", "explains to stakeholders", "clear in meetings"
    ],
    "written communication": [
        "written communication", "writes well", "clear emails",
        "good documentation", "professional writing", "concise reports"
    ],
    "giving feedback": [
        "giving feedback", "provides constructive criticism",
        "coaches through feedback", "shares observations openly"
    ],
    "receiving feedback": [
        "receiving feedback", "accepts criticism well", "open to feedback",
        "responds positively to critique", "non-defensive"
    ],

    # ── COLLABORATION ─────────────────────────────────────────────
    "teamwork": [
        "teamwork", "works well with others", "collaborative",
        "team player", "supports colleagues", "contributes to the team"
    ],
    "conflict resolution": [
        "conflict resolution", "resolves disagreements", "handles disputes",
        "mediates between parties", "de-escalates tension", "manages friction"
    ],
    "cross-functional collaboration": [
        "cross-functional collaboration", "works across teams",
        "partners with other departments", "coordinates with stakeholders"
    ],
    "empathy": [
        "empathy", "understands others feelings", "emotionally aware",
        "considers team morale", "compassionate", "supportive of peers"
    ],
    "relationship building": [
        "relationship building", "builds rapport", "earns trust",
        "maintains strong working relationships", "connects with colleagues"
    ],

    # ── DELIVERY & EXECUTION ──────────────────────────────────────
    "time management": [
        "time management", "meets deadlines", "delivers on time",
        "manages priorities", "punctual", "organized", "respects timelines"
    ],
    "productivity": [
        "productivity", "gets things done", "high output",
        "efficient worker", "delivers consistently", "results-oriented"
    ],
    "attention to detail": [
        "attention to detail", "thorough", "catches errors",
        "meticulous", "accurate work", "quality conscious", "precise"
    ],
    "goal achievement": [
        "goal achievement", "meets targets", "achieves objectives",
        "hits KPIs", "delivers results", "exceeds expectations"
    ],
    "project management": [
        "project management", "manages projects well", "coordinates deliverables",
        "tracks milestones", "keeps projects on track", "plans effectively"
    ],
    "prioritization": [
        "prioritization", "focuses on what matters", "manages workload",
        "handles multiple tasks", "knows what to tackle first"
    ],

    # ── THINKING & PROBLEM SOLVING ────────────────────────────────
    "problem solving": [
        "problem solving", "finds solutions", "resolves issues",
        "tackles challenges", "troubleshoots effectively", "overcomes obstacles"
    ],
    "critical thinking": [
        "critical thinking", "analytical", "thinks deeply",
        "questions assumptions", "evaluates situations carefully", "logical"
    ],
    "strategic thinking": [
        "strategic thinking", "sees the big picture", "thinks long term",
        "aligns work with strategy", "forward thinking", "visionary"
    ],
    "creativity": [
        "creativity", "innovative", "thinks outside the box",
        "brings new ideas", "creative solutions", "imaginative approach"
    ],
    "data-driven thinking": [
        "data-driven thinking", "uses data to decide", "evidence-based",
        "relies on metrics", "analytical approach", "backs decisions with data"
    ],

    # ── ADAPTABILITY & GROWTH ─────────────────────────────────────
    "adaptability": [
        "adaptability", "adapts to change", "flexible",
        "handles uncertainty", "adjusts quickly", "embraces new challenges"
    ],
    "learning agility": [
        "learning agility", "learns quickly", "picks up new skills fast",
        "eager to learn", "grows continuously", "develops new knowledge"
    ],
    "resilience": [
        "resilience", "bounces back", "handles pressure well",
        "stays calm under stress", "perseveres", "does not give up easily"
    ],
    "growth mindset": [
        "growth mindset", "seeks improvement", "open to learning",
        "embraces challenges", "treats failure as learning", "self-improving"
    ],

    # ── PROFESSIONALISM ───────────────────────────────────────────
    "work ethic": [
        "work ethic", "hardworking", "dedicated", "committed",
        "puts in effort", "goes the extra mile", "diligent",
        "working hard", "completes tasks", "gets the job done",
        "puts in the work", "great effort"
    ],
    "integrity": [
        "integrity", "honest", "trustworthy", "ethical",
        "acts with principles", "transparent", "does the right thing"
    ],
    "professionalism": [
        "professionalism", "conducts himself professionally",
        "conducts herself professionally", "represents the company well",
        "appropriate workplace behavior", "maintains standards"
    ],
    "punctuality": [
        "punctuality", "always on time", "never late",
        "respects others time", "arrives prepared", "timely"
    ],

    # ── TECHNICAL & DOMAIN ────────────────────────────────────────
    "technical skills": [
        "technical skills", "technically strong", "domain expertise",
        "deep knowledge", "subject matter expert", "skilled in tools"
    ],
    "digital literacy": [
        "digital literacy", "comfortable with technology",
        "uses software effectively", "tech savvy", "adapts to new tools"
    ],
    "knowledge sharing": [
        "knowledge sharing", "shares expertise", "documents learnings",
        "teaches the team", "spreads best practices", "contributes to wikis"
    ],
}
print(f"Skill list ready: {len(SKILL_ALIASES)} skills")

Skill list ready: 40 skills


In [ ]:
alias_texts = []
alias_to_skill = []

for skill, aliases in SKILL_ALIASES.items():
    for alias in aliases:
        alias_texts.append(alias)
        alias_to_skill.append(skill)

alias_embeddings = model.encode(alias_texts, convert_to_tensor=True)

print(f"Pre-computed {len(alias_texts)} alias embeddings")
print(f"Embedding matrix shape: {alias_embeddings.shape}")

Pre-computed 243 alias embeddings
Embedding matrix shape: torch.Size([243, 1024])


## Negation & Degradation Detection

Key fix: We now distinguish between:
1. **Direct negations** — 'lacks', 'failed to', 'did not', etc.
2. **Degradation phrases** — 'too slow', 'took too long', 'speed declined', etc.
3. **Contrast polarity** — clauses after 'but/however' are treated as likely negative

Also fixed: `is_negated()` no longer requires the skill word itself to appear in the sentence.
Instead it checks a **window of 60 characters around the matched alias** for negation signals.

In [ ]:
import re

# ── Direct negation words ──────────────────────────────────────────────────
NEGATION_PATTERNS = [
    # Core negations
    r"\blacks?\b", r"\bwithout\b", r"\bnever\b", r"\bpoor\b",
    r"\bweak\b", r"\bweakness\b",
    r"\bstruggles?\s+(with|to)\b", r"\bneeds?\s+to\s+improve\b",
    r"\bfails?\s+to\b", r"\bfailed\s+to\b",
    r"\bunable\s+to\b", r"\bdifficulty\s+(with)?\b", r"\bhas\s+difficulty\b",

    # Task/behavioral negations
    r"\bneglected\b", r"\bignored\b", r"\bmissed\b",
    r"\bskipped\b", r"\bavoided\b", r"\brefused\s+to\b",

    # Availability
    r"\bnot\s+available\b", r"\bunavailable\b",
    r"\balways\s+busy\b", r"\bnever\s+available\b",

    # Standard negations
    r"\bnot\s+able\s+to\b", r"\bwas\s+not\s+able\b",
    r"\bdidn'?t\b", r"\bdid\s+not\b",
    r"\bdoesn'?t\b", r"\bdoes\s+not\b",
    r"\bcan'?t\b", r"\bcannot\b", r"\bwon'?t\b",
    r"\bhard\s+to\b", r"\bdifficult\s+to\b",

    # Feedback/report negations
    r"\bnot\s+enough\b", r"\bcomplaints?\s+about\b",
    r"\bnegative\s+feedback\b", r"\braised\s+concerns?\b",
    r"\bhad\s+no\s+idea\b", r"\bno\s+idea\b",
    r"\bno\s+visibility\b", r"\bno\s+clue\b",
    r"\bwasn'?t\s+informed\b", r"\bnot\s+informed\b",
]

# ── NEW: Degradation / comparative negative patterns ──────────────────────
# These capture implicit negatives not covered by direct negation words
DEGRADATION_PATTERNS = [
    r"\btoo\s+slow\b",
    r"\btoo\s+long\b",          # 'took too long', 'it took him too long'
    r"\btook\s+too\s+long\b",
    r"\btakes?\s+too\s+long\b",
    r"\bvery\s+slow\b",
    r"\bspeed\s+(is|was|has)\s+(low|slow|poor|declining)\b",
    r"\bbecame?\s+low(er|ered)?\b",  # 'became lower'
    r"\bdeclin(ed|ing)\b",
    r"\bwors(e|ened)\b",
    r"\bbelow\s+(target|expectations?|average|standard)\b",
    r"\bmissed\s+(the\s+)?deadline\b",
    r"\boverdue\b",
    r"\bdelayed\b",
    r"\bnot\s+(fast|quick|efficient)\s+enough\b",
    r"\bslow\s+(pace|progress|performance|delivery)\b",
    r"\binsufficient\b",
    r"\binadequate\b",
    r"\broom\s+for\s+improvement\b",
    r"\bneeds?\s+improvement\b",
    r"\bunderperform(ing|ed|s)?\b",
    r"\bnot\s+meeting\s+(targets?|goals?|expectations?)\b",
    r"\bfell\s+short\b",
]

# Compile all patterns
_NEG_RE = [re.compile(p, re.IGNORECASE) for p in NEGATION_PATTERNS]
_DEG_RE = [re.compile(p, re.IGNORECASE) for p in DEGRADATION_PATTERNS]


def sentence_is_negative(sentence: str) -> bool:
    """
    Returns True if the sentence contains any direct negation
    or degradation signal — meaning it describes a WEAKNESS/GAP.

    This is the core fix: we check the WHOLE sentence for signals,
    not just a window around the skill alias.
    """
    s = sentence.lower()
    for pattern in _NEG_RE:
        if pattern.search(s):
            return True
    for pattern in _DEG_RE:
        if pattern.search(s):
            return True
    return False


print("Negation patterns loaded")

# Quick sanity checks
tests = [
    ("it took him too long to finish the tasks", True),
    ("she is great at communication", False),
    ("he lacks leadership skills", True),
    ("she is always available and responsive", False),
    ("the speed became lower", True),
    ("he missed the deadline", True),
    ("she is a great team player", False),
    ("did not respond to messages", True),
]
print("\nSanity checks:")
for text, expected in tests:
    result = sentence_is_negative(text)
    status = '✓' if result == expected else '✗ FAIL'
    print(f"  {status}  '{text}'  → {result}")

Negation patterns loaded

Sanity checks:
  ✓  'it took him too long to finish the tasks'  → True
  ✓  'she is great at communication'  → False
  ✓  'he lacks leadership skills'  → True
  ✓  'she is always available and responsive'  → False
  ✓  'the speed became lower'  → True
  ✓  'he missed the deadline'  → True
  ✓  'she is a great team player'  → False
  ✓  'did not respond to messages'  → True


In [ ]:
from wtpsplit import SaT
sat = SaT("sat-12l")
print("SaT model loaded")

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

SubwordXLMForTokenClassification LOAD REPORT from: segment-any-text/sat-12l
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SaT model loaded


## Clause Splitting with Polarity Tags

Key fix: Clauses that appear **after a contrast marker** ('but', 'however', 'although', etc.)
are tagged as `polarity='negative'` regardless of negation keywords.

This solves cases like:
- 'He is great but **his speed is slow**'
- 'She communicates well however **she is often unavailable**'

In [ ]:
CONTRAST_MARKERS = r'\b(but|however|although|though|yet|despite|nevertheless|that said|the problem is|on the other hand|unfortunately)\b'


def split_with_polarity(text: str) -> list:
    """
    Split text into clauses, each tagged with polarity:
      - 'positive' : clause appears before any contrast marker
      - 'contrast' : clause appears after a contrast marker (treated as negative context)
      - 'negative' : clause contains a direct negation or degradation signal

    Returns: list of dicts {text: str, polarity: str}
    """
    units = []

    # Split by contrast markers
    parts = re.split(CONTRAST_MARKERS, text, flags=re.IGNORECASE)
    # parts alternates: [before, marker, after_1, marker2, after_2, ...]
    # We skip the marker tokens (odd indices after first split)

    after_contrast = False
    for part in parts:
        part = part.strip()
        if not part:
            continue
        # Check if this part is a marker token
        if re.fullmatch(CONTRAST_MARKERS, part, flags=re.IGNORECASE):
            after_contrast = True
            continue
        if len(part) < 5:
            continue

        # Determine polarity
        if sentence_is_negative(part):
            polarity = 'negative'
        elif after_contrast:
            polarity = 'contrast'  # contrast context = likely gap
        else:
            polarity = 'positive'

        units.append({'text': part, 'polarity': polarity})

    # Also add the full sentence as an additional unit
    full_polarity = 'negative' if sentence_is_negative(text) else 'positive'
    units.append({'text': text.strip(), 'polarity': full_polarity})

    # Comma splits (but only if they're long enough to be meaningful)
    for part in text.split(','):
        part = part.strip()
        if len(part) > 15:  # avoid tiny fragments
            pol = 'negative' if sentence_is_negative(part) else 'positive'
            units.append({'text': part, 'polarity': pol})

    # Deduplicate by text
    seen = set()
    result = []
    for u in units:
        if u['text'].lower() not in seen:
            seen.add(u['text'].lower())
            result.append(u)

    return result


def split_into_clauses(text: str) -> list:
    """
    Top-level splitter. Returns list of {text, polarity} dicts.
    """
    if not text:
        return []
    text = text[0].upper() + text[1:]

    # First: sentence-level split via SaT
    segments = sat.split(text, threshold=0.0005)
    sentences = []
    for seg in segments:
        seg = seg.strip()
        if not seg:
            continue
        if sentences and (seg[0].islower() or len(seg.split()) < 2):
            sentences[-1] = sentences[-1] + ' ' + seg
        else:
            sentences.append(seg)

    # Then: expand each sentence into polarity-tagged clauses
    all_units = []
    for sentence in sentences:
        if len(sentence) > 8:
            all_units.extend(split_with_polarity(sentence))

    # Deduplicate
    seen = set()
    result = []
    for u in all_units:
        if u['text'].lower() not in seen:
            seen.add(u['text'].lower())
            result.append(u)

    return result


# Test
test_input = "Mohamed is getting more experienced at the tasks he is assigned to, but it took him too long to finish the tasks"
clauses = split_into_clauses(test_input)
print("Clause polarity analysis:")
for c in clauses:
    print(f"  [{c['polarity']:10s}]  {c['text']}")

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Clause polarity analysis:
  [positive  ]  Mohamed is getting more experienced at the tasks he is assigned to,
  [negative  ]  it took him too long to finish the tasks
  [negative  ]  Mohamed is getting more experienced at the tasks he is assigned to, but it took him too long to finish the tasks
  [positive  ]  Mohamed is getting more experienced at the tasks he is assigned to
  [negative  ]  but it took him too long to finish the tasks


## Core Extraction Functions

- `extract_skills()` — finds skills that are mentioned **positively** (polarity = positive)
- `get_skill_gaps()` — finds skills mentioned in **negative or contrast** clauses

Both skip ambiguous clauses where polarity is unclear.

In [ ]:
SIMILARITY_THRESHOLD = 0.58
PREFIX = "Represent this sentence for searching relevant passages: "


def _score_clauses(clauses: list) -> dict:
    """
    Internal: embed each clause and get top skill matches.
    Returns dict: skill -> {score, polarity}
    """
    skill_hits = {}  # skill -> {'score': float, 'polarity': str}

    for clause in clauses:
        text = clause['text']
        polarity = clause['polarity']

        prefixed = f"{PREFIX}{text}"
        emb = model.encode(prefixed, convert_to_tensor=True)
        scores = util.cos_sim(emb, alias_embeddings)[0]

        for i, score in enumerate(scores):
            score_val = float(score)
            if score_val < SIMILARITY_THRESHOLD:
                continue

            skill = alias_to_skill[i]

            # Update if this is the best score for this skill under this polarity
            if skill not in skill_hits or score_val > skill_hits[skill]['score']:
                skill_hits[skill] = {'score': round(score_val, 2), 'polarity': polarity}
            elif skill in skill_hits:
                # If same score but different polarity — prefer the negative signal
                if polarity in ('negative', 'contrast') and skill_hits[skill]['polarity'] == 'positive':
                    skill_hits[skill]['polarity'] = polarity

    return skill_hits


def extract_skills(review_text: str) -> list:
    """
    Returns skills that are clearly PRESENT (positive polarity only).
    Excludes any skill that also appears as a gap.
    """
    clauses = split_into_clauses(review_text)
    skill_hits = _score_clauses(clauses)

    found = {
        skill: info['score']
        for skill, info in skill_hits.items()
        if info['polarity'] == 'positive'
    }

    # Remove any skill that also appears as a gap
    gaps = {g['skill'] for g in get_skill_gaps(review_text)}
    found = {k: v for k, v in found.items() if k not in gaps}

    return sorted(
        [{'skill': s, 'confidence': sc} for s, sc in found.items()],
        key=lambda x: x['confidence'],
        reverse=True
    )


def get_skill_gaps(review_text: str) -> list:
    """
    Returns skills that are clearly MISSING or WEAK (negative/contrast polarity).
    """
    clauses = split_into_clauses(review_text)
    skill_hits = _score_clauses(clauses)

    gaps = {
        skill: info['score']
        for skill, info in skill_hits.items()
        if info['polarity'] in ('negative', 'contrast')
    }

    return sorted(
        [{'skill': s, 'confidence': sc} for s, sc in gaps.items()],
        key=lambda x: x['confidence'],
        reverse=True
    )


print("Functions defined.")

Functions defined.


## Test Cases

Test with the original failing example and several new ones to verify generalization.

In [ ]:
def run_analysis(review: str, label: str = ""):
    """Run and display skills analysis for a given review text."""
    print(f"{'='*60}")
    if label:
        print(f"TEST: {label}")
    print(f"INPUT: {review.strip()}")
    print()

    found = extract_skills(review)
    gaps = get_skill_gaps(review)

    print("SKILLS FOUND:")
    if found:
        for s in found:
            print(f"  {s['skill']:<30} confidence: {s['confidence']}")
    else:
        print("  (none)")

    print("\nSKILL GAPS:")
    if gaps:
        for g in gaps:
            print(f"  {g['skill']:<30} confidence: {g['confidence']}")
    else:
        print("  (none)")
    print()


# ── Test 1: Original failing case ─────────────────────────────────────────
run_analysis(
    "Mohamed is getting more experienced at the tasks he is assigned to, but it took him too long to finish the tasks",
    label="Original case — should show: learning agility found, time mgmt/productivity gap"
)

TEST: Original case — should show: learning agility found, time mgmt/productivity gap
INPUT: Mohamed is getting more experienced at the tasks he is assigned to, but it took him too long to finish the tasks

SKILLS FOUND:
  technical skills               confidence: 0.61
  leadership                     confidence: 0.59
  knowledge sharing              confidence: 0.59
  teamwork                       confidence: 0.58

SKILL GAPS:
  work ethic                     confidence: 0.65
  prioritization                 confidence: 0.61
  learning agility               confidence: 0.61
  delegation                     confidence: 0.6
  time management                confidence: 0.6
  productivity                   confidence: 0.6
  problem solving                confidence: 0.58



In [ ]:
# ── Test 2: Availability gap ───────────────────────────────────────────────
run_analysis(
    "She is always late to meetings and misses deadlines without notifying anyone in advance",
    label="Availability gap"
)

TEST: Availability gap
INPUT: She is always late to meetings and misses deadlines without notifying anyone in advance

SKILLS FOUND:
  (none)

SKILL GAPS:
  active listening               confidence: 0.59
  time management                confidence: 0.58



In [ ]:
# ── Test 3: Clear positive skills ─────────────────────────────────────────
run_analysis(
    "She is always late to meetings and misses deadlines without notifying anyone in advance",
    label="Purely positive review"
)

TEST: Purely positive review
INPUT: She is always late to meetings and misses deadlines without notifying anyone in advance

SKILLS FOUND:
  (none)

SKILL GAPS:
  active listening               confidence: 0.59
  time management                confidence: 0.58



In [ ]:
# ── Test 4: Mixed review ───────────────────────────────────────────────────
run_analysis(
    "Noha communicates well and builds strong relationships, however she struggles with prioritization and missed several deadlines this quarter",
    label="Mixed — communication positive, prioritization/time management gap"
)

TEST: Mixed — communication positive, prioritization/time management gap
INPUT: Noha communicates well and builds strong relationships, however she struggles with prioritization and missed several deadlines this quarter

SKILLS FOUND:
  relationship building          confidence: 0.62

SKILL GAPS:
  time management                confidence: 0.7
  prioritization                 confidence: 0.62
  productivity                   confidence: 0.6
  punctuality                    confidence: 0.6
  accountability                 confidence: 0.59
  project management             confidence: 0.59
  work ethic                     confidence: 0.58



In [ ]:
# ── Test 5: Degradation without explicit negation ──────────────────────────
run_analysis(
    "His output quality declined over the last quarter and performance fell short of targets",
    label="Degradation signals only (no 'not', 'lacks', etc.)"
)

TEST: Degradation signals only (no 'not', 'lacks', etc.)
INPUT: His output quality declined over the last quarter and performance fell short of targets

SKILLS FOUND:
  (none)

SKILL GAPS:
  (none)



In [ ]:
# ── Test 6: Feedback/visibility gap ───────────────────────────────────────
run_analysis(
    "im working with noha and she is great person to work with she always support me",
    label=")"
)

TEST: )
INPUT: im working with noha and she is great person to work with she always support me

SKILLS FOUND:
  relationship building          confidence: 0.66
  teamwork                       confidence: 0.62
  receiving feedback             confidence: 0.6
  empathy                        confidence: 0.59
  knowledge sharing              confidence: 0.59
  active listening               confidence: 0.58
  work ethic                     confidence: 0.58

SKILL GAPS:
  (none)



In [ ]:
TEST_SET = [
    {
        "review": "Sarah consistently leads her team through difficult projects and mentors junior staff effectively. She communicates clearly in meetings and always delivers on time.",
        "expected_skills": ["leadership", "mentoring", "communication", "time management"],
        "expected_gaps": []
    },
    {
        "review": "Ahmed works hard and completes all his tasks diligently. However, he struggled to adapt when the project requirements changed suddenly.",
        "expected_skills": ["work ethic", "productivity"],
        "expected_gaps": ["adaptability"]
    },
    {
        "review": "It is hard to communicate with Sara when I need her. She is unavailable most of the day and does not respond to messages.",
        "expected_skills": [],
        "expected_gaps": ["communication", "availability"]
    },
    {
        "review": "John is a great team player and builds strong relationships with his colleagues. He is empathetic and resolves conflicts calmly.",
        "expected_skills": ["teamwork", "relationship building", "empathy", "conflict resolution"],
        "expected_gaps": []
    },
    {
        "review": "Hashem neglected several meetings this month and did not follow through on his commitments. He lacks accountability.",
        "expected_skills": [],
        "expected_gaps": ["punctuality", "accountability", "professionalism"]
    },
    {
        "review": "Lena brings creative solutions to every challenge. She thinks strategically and uses data to back her decisions.",
        "expected_skills": ["creativity", "strategic thinking", "data-driven thinking", "problem solving"],
        "expected_gaps": []
    },
    {
        "review": "Mike always arrives prepared and on time. He is professional in all client interactions and maintains high standards.",
        "expected_skills": ["punctuality", "professionalism", "work ethic"],
        "expected_gaps": []
    },
    {
        "review": "Nora picks up new tools and skills very quickly. She is eager to learn and continuously grows in her role.",
        "expected_skills": ["learning agility", "growth mindset", "adaptability"],
        "expected_gaps": []
    },
    {
        "review": "Omar fails to meet deadlines repeatedly. He does not prioritize his work and struggles with managing multiple tasks.",
        "expected_skills": [],
        "expected_gaps": ["time management", "prioritization", "productivity"]
    },
    {
        "review": "Fatima presents confidently to senior stakeholders and writes clear and concise project reports.",
        "expected_skills": ["presentation", "written communication", "communication"],
        "expected_gaps": []
    },
    {
        "review": "David delegates tasks well and empowers his team. He makes sound decisions even under pressure.",
        "expected_skills": ["delegation", "decision making", "leadership", "resilience"],
        "expected_gaps": []
    },
    {
        "review": "The employee shares knowledge openly with the team and documents her work thoroughly.",
        "expected_skills": ["knowledge sharing", "written communication"],
        "expected_gaps": []
    },
    {
        "review": "He is technically very strong and adapts to new tools quickly. He is a subject matter expert in his domain.",
        "expected_skills": ["technical skills", "digital literacy", "adaptability"],
        "expected_gaps": []
    },
    {
        "review": "She accepts feedback well and uses it to improve. She never gets defensive when criticized.",
        "expected_skills": ["receiving feedback", "growth mindset"],
        "expected_gaps": []
    },
    {
        "review": "He refused to collaborate with the other department and avoided cross-team meetings entirely.",
        "expected_skills": [],
        "expected_gaps": ["cross-functional collaboration", "teamwork"]
    },
    {
        "review": "Yasmin is honest and transparent in all her dealings. The team trusts her completely.",
        "expected_skills": ["integrity", "professionalism"],
        "expected_gaps": []
    },
    {
        "review": "He bounces back quickly from setbacks and stays calm when things go wrong. Very resilient under pressure.",
        "expected_skills": ["resilience", "adaptability"],
        "expected_gaps": []
    },
    {
        "review": "She took the initiative to redesign the onboarding process without being asked. A true self-starter.",
        "expected_skills": ["initiative", "creativity", "work ethic"],
        "expected_gaps": []
    },
    {
        "review": "He listens carefully in meetings and always makes sure he understands before responding.",
        "expected_skills": ["active listening", "communication"],
        "expected_gaps": []
    },
    {
        "review": "She consistently hits her KPIs and exceeds the targets set for her each quarter.",
        "expected_skills": ["goal achievement", "productivity", "work ethic"],
        "expected_gaps": []
    },
    {
        "review": "He is a strong communicator but lacks critical thinking when solving complex problems.",
        "expected_skills": ["communication"],
        "expected_gaps": ["critical thinking", "problem solving"]
    },
    {
        "review": "She works hard but does not share her knowledge with the team and avoids helping others.",
        "expected_skills": ["work ethic"],
        "expected_gaps": ["knowledge sharing", "teamwork"]
    },
    {
        "review": "He cannot manage his time well and always misses deadlines. However he is very creative.",
        "expected_skills": ["creativity"],
        "expected_gaps": ["time management", "productivity"]
    },
    {
        "review": "She gives great presentations but does not accept feedback well and gets defensive easily.",
        "expected_skills": ["presentation", "communication"],
        "expected_gaps": ["receiving feedback"]
    },
    {
        "review": "He is punctual and professional but struggles with making decisions under pressure.",
        "expected_skills": ["punctuality", "professionalism"],
        "expected_gaps": ["decision making", "resilience"]
    },

    # ── GAP-ONLY REVIEWS ─────────────────────────────────────────
    {
        "review": "He never takes responsibility for his mistakes and always blames others when things go wrong.",
        "expected_skills": [],
        "expected_gaps": ["accountability", "integrity"]
    },
    {
        "review": "She does not respond to emails or messages for days. The team cannot rely on her being reachable.",
        "expected_skills": [],
        "expected_gaps": ["communication", "availability"]
    },
    {
        "review": "He struggles to work with other departments and refuses to attend cross-functional meetings.",
        "expected_skills": [],
        "expected_gaps": ["cross-functional collaboration", "teamwork"]
    },
    {
        "review": "She cannot handle pressure and breaks down whenever there is a tight deadline.",
        "expected_skills": [],
        "expected_gaps": ["resilience", "time management"]
    },
    {
        "review": "He never shares what he knows with the rest of the team. Knowledge stays with him alone.",
        "expected_skills": [],
        "expected_gaps": ["knowledge sharing", "teamwork"]
    },
    {
        "review": "She is unable to make decisions without asking for approval on every small detail.",
        "expected_skills": [],
        "expected_gaps": ["decision making", "initiative"]
    },
    {
        "review": "He ignored feedback from his manager repeatedly and never made any effort to improve.",
        "expected_skills": [],
        "expected_gaps": ["receiving feedback", "growth mindset"]
    },
    {
        "review": "She lacks strategic thinking and cannot see how her work connects to the bigger picture.",
        "expected_skills": [],
        "expected_gaps": ["strategic thinking", "critical thinking"]
    },
    {
        "review": "He avoided all client presentations and did not contribute to any written reports this quarter.",
        "expected_skills": [],
        "expected_gaps": ["presentation", "written communication"]
    },
    {
        "review": "She is always late to meetings and misses deadlines without notifying anyone in advance.",
        "expected_skills": [],
        "expected_gaps": ["punctuality", "time management", "communication"]
    },
]

print(f"Test set ready: {len(TEST_SET)} reviews")
print(f"  — Skills only : {sum(1 for x in TEST_SET if x['expected_skills'] and not x['expected_gaps'])}")
print(f"  — Gaps only   : {sum(1 for x in TEST_SET if x['expected_gaps'] and not x['expected_skills'])}")
print(f"  — Mixed       : {sum(1 for x in TEST_SET if x['expected_skills'] and x['expected_gaps'])}")
print(f"  — Clean       : {sum(1 for x in TEST_SET if not x['expected_skills'] and not x['expected_gaps'])}")

Test set ready: 35 reviews
  — Skills only : 15
  — Gaps only   : 14
  — Mixed       : 6
  — Clean       : 0


In [ ]:
def calculate_metrics(true_positives, false_positives, false_negatives):
    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1

# Initialize counters for skills
tp_skills = 0
fp_skills = 0
fn_skills = 0

# Initialize counters for gaps
tp_gaps = 0
fp_gaps = 0
fn_gaps = 0

for i, test_case in enumerate(TEST_SET):
    review = test_case['review']
    expected_skills = set(test_case['expected_skills'])
    expected_gaps = set(test_case['expected_gaps'])

    # Evaluate skills
    found_skills_dicts = extract_skills(review)
    found_skills = {s['skill'] for s in found_skills_dicts}

    tp_skills += len(expected_skills.intersection(found_skills))
    fp_skills += len(found_skills.difference(expected_skills))
    fn_skills += len(expected_skills.difference(found_skills))

    # Evaluate gaps
    found_gaps_dicts = get_skill_gaps(review)
    found_gaps = {g['skill'] for g in found_gaps_dicts}

    tp_gaps += len(expected_gaps.intersection(found_gaps))
    fp_gaps += len(found_gaps.difference(expected_gaps))
    fn_gaps += len(expected_gaps.difference(found_gaps))

print("\n" + "="*60)
print("Overall Model Evaluation Results")
print("="*60)

# Calculate and print metrics for Skills
precision_skills, recall_skills, f1_skills = calculate_metrics(tp_skills, fp_skills, fn_skills)
print("\nSKILL DETECTION:")
print(f"  True Positives:  {tp_skills}")
print(f"  False Positives: {fp_skills}")
print(f"  False Negatives: {fn_skills}")
print(f"  Precision:       {precision_skills:.2f}")
print(f"  Recall:          {recall_skills:.2f}")
print(f"  F1-Score:        {f1_skills:.2f}")

# Calculate and print metrics for Gaps
precision_gaps, recall_gaps, f1_gaps = calculate_metrics(tp_gaps, fp_gaps, fn_gaps)
print("\nSKILL GAP DETECTION:")
print(f"  True Positives:  {tp_gaps}")
print(f"  False Positives: {fp_gaps}")
print(f"  False Negatives: {fn_gaps}")
print(f"  Precision:       {precision_gaps:.2f}")
print(f"  Recall:          {recall_gaps:.2f}")
print(f"  F1-Score:        {f1_gaps:.2f}")



Overall Model Evaluation Results

SKILL DETECTION:
  True Positives:  33
  False Positives: 151
  False Negatives: 20
  Precision:       0.18
  Recall:          0.62
  F1-Score:        0.28

SKILL GAP DETECTION:
  True Positives:  16
  False Positives: 65
  False Negatives: 25
  Precision:       0.20
  Recall:          0.39
  F1-Score:        0.26


### Overall Accuracy Calculation

To calculate overall accuracy, we need the number of True Negatives (TN) in addition to the True Positives (TP), False Positives (FP), and False Negatives (FN) already calculated.

We can derive TN by considering the total number of possible (review, skill) pairs in our test set. For each review, each skill can either be correctly identified as present (TP), incorrectly identified as present (FP), present but not identified (FN), or correctly identified as not present (TN).

In [ ]:
# Total number of unique skills in our dictionary
num_skills = len(SKILL_ALIASES)

# Total number of test cases
num_test_cases = len(TEST_SET)

# Total possible (review, skill) predictions across the entire test set
total_possible_predictions = num_skills * num_test_cases

print(f"Total unique skills: {num_skills}")
print(f"Total test cases: {num_test_cases}")
print(f"Total possible (review, skill) predictions: {total_possible_predictions}")

# --- Calculate Accuracy for SKILL DETECTION ---
# From previous output:
# tp_skills = 33
# fp_skills = 151
# fn_skills = 20

tn_skills = total_possible_predictions - (tp_skills + fp_skills + fn_skills)
accuracy_skills = (tp_skills + tn_skills) / total_possible_predictions

print("\n--- SKILL DETECTION ACCURACY ---")
print(f"  True Negatives (derived): {tn_skills}")
print(f"  Accuracy: {accuracy_skills:.2f}")

# --- Calculate Accuracy for SKILL GAP DETECTION ---
# From previous output:
# tp_gaps = 16
# fp_gaps = 65
# fn_gaps = 25

tn_gaps = total_possible_predictions - (tp_gaps + fp_gaps + fn_gaps)
accuracy_gaps = (tp_gaps + tn_gaps) / total_possible_predictions

print("\n--- SKILL GAP DETECTION ACCURACY ---")
print(f"  True Negatives (derived): {tn_gaps}")
print(f"  Accuracy: {accuracy_gaps:.2f}")

Total unique skills: 40
Total test cases: 35
Total possible (review, skill) predictions: 1400

--- SKILL DETECTION ACCURACY ---
  True Negatives (derived): 1196
  Accuracy: 0.88

--- SKILL GAP DETECTION ACCURACY ---
  True Negatives (derived): 1294
  Accuracy: 0.94
